# Reg 28 Table 2 (Infrastructure)

In [11]:
# libraries, libraries!

%run utilities.ipynb
start_time0 = time.time()
start_time  = time.time()
print(f'Importing libraries ...')

import pandas as pd
import numpy as np
import openpyxl, os, math

print(f'Importing libraries completed: {timediff(start_time, time.time())}', '\n')

Importing libraries ...
Importing libraries completed: 0.0sec 



In [12]:
# set up paths as strings
start_time = time.time()
print(f'Setting up paths ...')

pthPy    = r'P:\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm'     # variables stored here
pth_rpt  = r'P:\Investment Operations\GRC\Compliance\Reg28 and Reg30 Reporting'
pth_tpl  = r'P:\Working Folders\Hilton\W\!Reg28 SchIB.xlsm'                     # the Table 2 template
pth_Test = r'P:\Working Folders\Hilton\W\Reg_Tests'                             # where to save in Test folder
pth_nl   = r'\\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Daily\2A - Fund Codes, Breach Register.xlsx'

print(f'Setting up paths completed: {timediff(start_time, time.time())}', '\n')

Setting up paths ...
Setting up paths completed: 0.0sec 



In [13]:
# get report variables
start_time = time.time()
print(f'Getting report variables ...')

p        = pd.read_excel(pthPy, sheet_name = 'r28_tbl2', index_col = None,  header = 0, \
                  usecols = 'A,D,F').dropna(subset = ['Fund']) # py reports sheet
fnd      = p.iat[0,0].upper()
rpt_Type = p.iat[0,1]
k        = p.iat[0,2]
rptDate  = prior_month_end() if math.isnan(k) else k # prior working ay or report date override; has type datetime.datetime()
print(' ',fnd, rptDate.strftime("%d%b%Y"), rpt_Type)

# fund long name lookup
nl       = pd.read_excel(pth_nl, sheet_name = 'Funds', index_col = None,  header = 0, \
                  usecols = 'A,B').dropna(subset = ['Fund Code']) # fund long name lookup

# Table 1 to Table 2 category translation
t1t2     = pd.read_excel(pth_tpl, sheet_name = 'Static', index_col = None,  header = 0, \
                  usecols = 'A,E').dropna(subset = ['Table 2'])

# Table 2 file name
fnm      = os.path.join(pth_Test,f'{fnd} Reg28 Table2 {rptDate.strftime("%d%b%Y")}.xlsx')

print(f'Getting report variables completed: {timediff(start_time, time.time())}', '\n')

Getting report variables ...
  PPSBAL_C 31Mar2025 Reg28
Getting report variables completed: 2.4sec 



In [14]:
# regulation categorised sheet
start_time = time.time()
print(f'Getting Table 2 items ...')

shtr          = pd.read_excel(os.path.join(pth_rpt, f'{fnd} {rpt_Type} {rptDate.strftime("%d%b%Y")}.xlsx'), \
                        index_col = None,  header = 0, usecols = 'A:K').dropna(subset = ['Entity Name']) # reg categorised sheet

shtr['Tbl2']  = shtr.apply(lambda row: t1t2[t1t2["Reg 28 Classification"] == row["Reg 28 Classification"]].iat[0,1] \
                          if row['Infrastructure'] == "11(b)" else float("nan"), axis = 1) # add a translated Table 2 column

shtr['Instr'] = shtr.apply(lambda row: '   ~ ' + row['Primary Asset ID'] + ' - ' + row['i Issue Name'], \
                           axis = 1) # instrument ID and name

print(f'Getting Table 2 items completed: {timediff(start_time, time.time())}', '\n')

Getting Table 2 items ...
Getting Table 2 items completed: 0.5sec 



In [15]:
# update cell values with openpyxl
start_time = time.time()
print(f'Importing summary totals ...')

pth_tmpl   = r'P:\Working Folders\Hilton\W\!Reg28_Tbl2.xlsx'
wb1        = openpyxl.load_workbook(pth_tmpl) # open the Table 2 template
sh1        = wb1['Tbl2'] # instead of sh1 = wb1.active

# https://stackoverflow.com/questions/70642758/how-to-read-a-specific-worksheet-with-openpyxl-instead-of-active-sheet
#wb1        = openpyxl.load_workbook(pthPy) # open the Table 2 template
#sh1        = wb1['r28_tbl2_tmpl']

sh1['A2']  = f'{nl[nl["Fund Code"] == fnd].iat[0,1]} ({fnd})'
sh1['A5']  = f'As at {rptDate.strftime("%d %B %Y")}'
sh1[item_row(sh1, "Total", 6)][1].value = shtr[shtr['Infrastructure'] == "11(b)"]['Percentage of Market Value'].sum()
sh1[item_row(sh1, "Total", 6)][2].value = shtr[shtr['Infrastructure'] == "11(b)"]['End Market Value'].sum()
sh1[item_row(sh1, "Fund Net Asset Value", 6)][2].value = shtr['End Market Value'].sum()
sh1[item_row(sh1, "2.2",   6)][1].value = shtr[shtr['Tbl2'] == '2.2.1']['Percentage of Market Value'].sum() + shtr[shtr['Tbl2'] == '2.2.2']['Percentage of Market Value'].sum()
sh1[item_row(sh1, "2.2",   6)][2].value = shtr[shtr['Tbl2'] == '2.2.1']['End Market Value'          ].sum() + shtr[shtr['Tbl2'] == '2.2.2']['End Market Value'          ].sum()
sh1[item_row(sh1, "2.",    6)][1].value = shtr[shtr['Tbl2'] == '2.2.1']['Percentage of Market Value'].sum() + shtr[shtr['Tbl2'] == '2.2.2']['Percentage of Market Value'].sum()
sh1[item_row(sh1, "2.",    6)][2].value = shtr[shtr['Tbl2'] == '2.2.1']['End Market Value'          ].sum() + shtr[shtr['Tbl2'] == '2.2.2']['End Market Value'          ].sum()
sh1[item_row(sh1, "3.",    6)][1].value = shtr[shtr['Tbl2'] == '3.1'  ]['Percentage of Market Value'].sum() + shtr[shtr['Tbl2'] == '3.2'  ]['Percentage of Market Value'].sum()
sh1[item_row(sh1, "3.",    6)][2].value = shtr[shtr['Tbl2'] == '3.1'  ]['End Market Value'          ].sum() + shtr[shtr['Tbl2'] == '3.2'  ]['End Market Value'          ].sum()
sh1[item_row(sh1, "3.",    6)][1].value = shtr[shtr['Tbl2'] == '3.1'  ]['Percentage of Market Value'].sum() + shtr[shtr['Tbl2'] == '3.2'  ]['Percentage of Market Value'].sum()
sh1[item_row(sh1, "3.",    6)][2].value = shtr[shtr['Tbl2'] == '3.1'  ]['End Market Value'          ].sum() + shtr[shtr['Tbl2'] == '3.2'  ]['End Market Value'          ].sum()
sh1[item_row(sh1, "4.",    6)][1].value = shtr[shtr['Tbl2'] == '4.'   ]['Percentage of Market Value'].sum() + shtr[shtr['Tbl2'] == '4.'   ]['Percentage of Market Value'].sum()
sh1[item_row(sh1, "4.",    6)][2].value = shtr[shtr['Tbl2'] == '4.'   ]['End Market Value'          ].sum() + shtr[shtr['Tbl2'] == '4.'   ]['End Market Value'          ].sum()
sh1[item_row(sh1, "5.",    6)][1].value = shtr[shtr['Tbl2'] == '5.'   ]['Percentage of Market Value'].sum() + shtr[shtr['Tbl2'] == '5.'   ]['Percentage of Market Value'].sum()
sh1[item_row(sh1, "5.",    6)][2].value = shtr[shtr['Tbl2'] == '5.'   ]['End Market Value'          ].sum() + shtr[shtr['Tbl2'] == '5.'   ]['End Market Value'          ].sum()
sh1[item_row(sh1, "6.",    6)][1].value = shtr[shtr['Tbl2'] == '6.'   ]['Percentage of Market Value'].sum() + shtr[shtr['Tbl2'] == '6.'   ]['Percentage of Market Value'].sum()
sh1[item_row(sh1, "6.",    6)][2].value = shtr[shtr['Tbl2'] == '6.'   ]['End Market Value'          ].sum() + shtr[shtr['Tbl2'] == '6.'   ]['End Market Value'          ].sum()
sh1[item_row(sh1, "8.",    6)][1].value = shtr[shtr['Tbl2'] == '8.'   ]['Percentage of Market Value'].sum() + shtr[shtr['Tbl2'] == '8.'   ]['Percentage of Market Value'].sum()
sh1[item_row(sh1, "8.",    6)][2].value = shtr[shtr['Tbl2'] == '8.'   ]['End Market Value'          ].sum() + shtr[shtr['Tbl2'] == '8.'   ]['End Market Value'          ].sum()
sh1[item_row(sh1, "9.",    6)][1].value = shtr[shtr['Tbl2'] == '9.'   ]['Percentage of Market Value'].sum() + shtr[shtr['Tbl2'] == '9.'   ]['Percentage of Market Value'].sum()
sh1[item_row(sh1, "9.",    6)][2].value = shtr[shtr['Tbl2'] == '9.'   ]['End Market Value'          ].sum() + shtr[shtr['Tbl2'] == '9.'   ]['End Market Value'          ].sum()
sh1[item_row(sh1, "10.",   6)][1].value = shtr[shtr['Tbl2'] == '10.'  ]['Percentage of Market Value'].sum() + shtr[shtr['Tbl2'] == '10.'  ]['Percentage of Market Value'].sum()
sh1[item_row(sh1, "10.",   6)][2].value = shtr[shtr['Tbl2'] == '10.'  ]['End Market Value'          ].sum() + shtr[shtr['Tbl2'] == '10.'  ]['End Market Value'          ].sum()
sh1[item_row(sh1, "11.2",  6)][1].value = shtr[shtr['Tbl2'] == '11.2' ]['Percentage of Market Value'].sum() + shtr[shtr['Tbl2'] == '11.2' ]['Percentage of Market Value'].sum()
sh1[item_row(sh1, "11.2",  6)][2].value = shtr[shtr['Tbl2'] == '11.2' ]['End Market Value'          ].sum() + shtr[shtr['Tbl2'] == '11.2' ]['End Market Value'          ].sum()

sh1.title  = f'{fnd} Table2 {rptDate.strftime("%d%b%Y")}'

print(f'Importing summary totals completed: {timediff(start_time, time.time())}', '\n')

Importing summary totals ...
Importing summary totals completed: 0.1sec 



In [16]:
# update sheet with Table 2 values
start_time = time.time()
print(f'Importing Table 2 values ...')

#z     = pd.read_excel(pthPy, sheet_name = 'r28_tbl2_tmpl', index_col = None,  header = 6, usecols = 'E').dropna() # Table 2 categories
pth_tmpl = r'P:\Working Folders\Hilton\W\!Reg28_Tbl2.xlsx'
z     = pd.read_excel(pth_tmpl, sheet_name = 'Tbl2', index_col = None,  \
                      header = 6, usecols = 'E').dropna() # Table 2 categories

for item in z["Item"]:
    k = shtr.loc[(shtr['Infrastructure'] == "11(b)") & (shtr["Tbl2"] == item), \
             ["Percentage of Market Value", "End Market Value",\
              "Instr", "Issuer"]].sort_values("Issuer", axis = 0) # infrastructure instruments and issuers
    
    # https://stackoverflow.com/questions/32059397/pandas-groupby-without-turning-grouped-by-column-into-index
    m = k.groupby("Issuer", as_index = False)[['Percentage of Market Value','End Market Value']].sum() # Issuers
    
    # https://www.boardinfinity.com/blog/learn-about-reset-index-pandas/#:~:text=To%20reset%20the%20index%20on,causes%20it%20to%20return%20Nothing.
    k.reset_index(inplace = True, drop = True) 
    
    if len(m) > 0:
        sh1[item_row(sh1, item, 6)][1].value = shtr[shtr['Tbl2'] == item]['Percentage of Market Value'].sum()
        sh1[item_row(sh1, item, 6)][2].value = shtr[shtr['Tbl2'] == item]['End Market Value'          ].sum()        
        spaces = 1
        sh1.insert_rows(item_row(sh1, item, 6), spaces) # insert row location, number of rows to insert        
        for index, row in m.iterrows():
            row_insert = item_row(sh1, item, 6) + 1 + index + spaces * index
            #print(f'insert rows from row {row_insert} and insert text at row {row_insert + spaces}')
            sh1.insert_rows(row_insert, amount = spaces + 1) # insert row location, number of rows to insert  
            sh1[row_insert + spaces][1].value = row['Percentage of Market Value']      
            sh1[row_insert + spaces][2].value = row['End Market Value']    
            sh1[row_insert + spaces][3].value = row['Issuer']  
        
            #for index, row in k.iterrows():
                #sh1[item_row(sh1, item) + index + 2][1].value = row['Percentage of Market Value']      
                #sh1[item_row(sh1, item) + index + 2][2].value = row['End Market Value']    
                #sh1[item_row(sh1, item) + index + 2][3].value = row['Instr']   
                
print(f'Importing Table 2 values completed: {timediff(start_time, time.time())}', '\n')

Importing Table 2 values ...
Importing Table 2 values completed: 0.1sec 



In [17]:
# formatting
start_time = time.time()
print(f'Formatting the sheet ...')

from openpyxl.styles import Alignment
from openpyxl.styles.borders import Border, Side
row1 = item_row(sh1, "1.", 6)
rows = item_row(sh1, "TTLb", 6)
for row in range(row1, rows):
    #https://stackoverflow.com/questions/49525545/openpyxl-formatting-cell-with-decimal
    sh1["B{}".format(row)].number_format = '_-* #,##0.00_-;-* #,##0.00_-;_-* "-"_-;_-@_-'
    sh1["C{}".format(row)].number_format = '_-* #,##0.00_-;-* #,##0.00_-;_-* "-"_-;_-@_-'

for cell in sh1['A']: #https://stackoverflow.com/questions/24917201/applying-borders-to-a-cell-in-openpyxl
    for row in range(row1, rows):
        cell.border = Border(left = Side(style = 'thin'), right = Side(style = 'thin'))  
    
for cell in sh1['B']: #https://stackoverflow.com/questions/26671581/horizontal-text-alignment-in-openpyxl
    for row in range(row1, rows):
        cell.alignment = Alignment(vertical = 'top')
        cell.border = Border(left = Side(style = 'thin'), right = Side(style = 'thin'))

for cell in sh1['C']: #https://stackoverflow.com/questions/26671581/horizontal-text-alignment-in-openpyxl
    for row in range(row1, rows):
        cell.alignment = Alignment(vertical = 'top')
        cell.border = Border(left = Side(style = 'thin'), right = Side(style = 'thin'))        
        
for cell in sh1['D']: #https://stackoverflow.com/questions/38619471/iterate-through-all-rows-in-specific-column-openpyxl
    for row in range(row1, rows):
        cell.alignment = Alignment(wrap_text = True, vertical = 'top')
        cell.border = Border(left = Side(style = 'thin'), right = Side(style = 'thin'))

# table titles alignment
for col in range(1, 5):
        sh1.cell(item_row(sh1, "Item", 6), column = col).alignment = Alignment(wrap_text = True,
                                                                             vertical = 'top',
                                                                           horizontal = 'center')
        
for col in range(2, 4):
        sh1.cell(item_row(sh1, "TTLb", 6), column = col).alignment = Alignment(wrap_text = True,
                                                                             vertical = 'top',
                                                                           horizontal = 'center')    
        
# borders
item        = pd.read_excel(pth_tmpl, sheet_name = 'Tbl2', index_col = None, header = None, names = ['Items'], \
                            usecols = 'F').dropna()
item['RwN'] = item['Items'].apply(lambda row: item_row(sh1, row, 6))
rwn = item['RwN'].tolist() # list of row numbers requiring top borders
for itm in rwn:
    for col in range(1, 5):
        sh1.cell(row = itm, column = col).border = Border(left  = Side(style = 'thin'),
                                                          right = Side(style = 'thin'),
                                                          top   = Side(style = 'thin')) 
        
for col in range(1,4): # 'Total' bottom border
    sh1.cell(row = item_row(sh1, "Ttlb", 6), column = col).border = Border(left   = Side(style = 'thin'),
                                                                        right  = Side(style = 'thin'),
                                                                        top    = Side(style = 'thin'),
                                                                        bottom = Side(style = 'thin'))

for col in range(1, 5):
    for rw in range(item_row(sh1, "Ttlb", 6) + 1, item_row(sh1, "Ttlb", 6) + 3):
        sh1.cell(row = rw, column = col).border = Border(left   = Side(border_style = None),
                                                         right  = Side(border_style = None),
                                                         top    = Side(border_style = None),
                                                         bottom = Side(border_style = None))
    
for col in range(2,4): # 'Fund Net Asset Value' top and bottom borders
    sh1.cell(row = item_row(sh1, "Fund Net Asset Value", 6), column = col).border = Border(top    = Side(style = 'thin'),
                                                                                       bottom  = Side(style = 'double'))
#https://openpyxl.readthedocs.io/en/latest/api/openpyxl.styles.borders.html

for rw in range(item_row(sh1, "Ttlb", 6), item_row(sh1, "Ttlb", 6) + 3):
    sh1.cell(row = rw, column = 4).border = Border(right = Side(border_style = None),
                                                  top    = Side(style = 'thin'))

for rw in range(item_row(sh1, "Ttlb", 6) + 1, item_row(sh1, "Ttlb", 6) + 3):
    sh1.cell(row = rw, column = 4).border = Border(top   = Side(border_style = None),
                                                  bottom = Side(border_style = None))

print(f'Formatting the sheet completed: {timediff(start_time, time.time())}', '\n')

Formatting the sheet ...
Formatting the sheet completed: 1.2sec 



In [18]:
# remove guide columns - https://openpyxl.readthedocs.io/en/stable/editing_worksheets.html
start_time = time.time()
print(f'Removing extraneous columns ...')

sh1.delete_cols(5, 2) # delete columns E:F

print(f'Removing extraneous columns completed: {timediff(start_time, time.time())}', '\n')

Removing extraneous columns ...
Removing extraneous columns completed: 0.0sec 



In [19]:
# save the file
start_time = time.time()
print(f'Saving {fnd} Reg28 Table2 {rptDate.strftime("%d%b%Y")}.xlsx in {pth_Test} ...')

wb1.save(    fnm)
wb1.close
open_xl_file(fnm)
print(f'Saving {fnd} Reg28 Table2 {rptDate.strftime("%d%b%Y")}.xlsx in {pth_Test} completed: {timediff(start_time, time.time())}', '\n')
print(f'Roundtrip time: {timediff(start_time0, time.time())}')

Saving PPSBAL_C Reg28 Table2 31Mar2025.xlsx in P:\Working Folders\Hilton\W\Reg_Tests ...
Saving PPSBAL_C Reg28 Table2 31Mar2025.xlsx in P:\Working Folders\Hilton\W\Reg_Tests completed: 0.9sec 

Roundtrip time: 5.4sec


In [20]:
# python reports   P:\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm
# peporting folder P:\Investment Operations\GRC\Compliance\Reg28 and Reg30 Reporting
# Table 2 template P:\Working Folders\Hilton\W\!Reg28_Tbl2.xlsx
# test folder      P:\Working Folders\Hilton\W\Reg_Tests